# Tea Trend Analysis
Here is where we will analyze the tea data you made and formatted.

In [ ]:
import numpy as np
from datascience import *
import math as m
from EDS import *
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from bestfit import *

import random

This page is designed to work with any of the three substances (Caffeine, Ninhydrin, Fluoride); import the table that contains data on the substance you worked on.

In [ ]:
tea_data = Table.read_table('...')
tea_data

Let's look at the concentrations for both kinds of tea.

In [ ]:
black_tea = tea_data.where('Type', are.equal_to('BT'))
green_tea = tea_data.where('Type', are.equal_to('GT'))

plt.scatter(black_tea['Temperature'], black_tea['Concentration (M)'], label = 'Black')
plt.scatter(green_tea['Temperature'], green_tea['Concentration (M)'], label = 'Green')

plt.xlabel('Temperature (°C)')

plt.ylabel('Concentration (M)')

plt.title('Concentration as a function of Temperature')

plt.legend()

plt.show()

In [ ]:
plt.scatter(black_tea['Steep Time'], black_tea['Concentration (M)'], label = 'Black')
plt.scatter(green_tea['Steep Time'], green_tea['Concentration (M)'], label = 'Green')

plt.xlabel('Steep Time (min)')

plt.ylabel('Concentration (M)')

plt.title('Concentration as a function of Steep Time')

plt.legend()

plt.show()

What do you notice about the two teas? What happens as the Temperature or Steep Time increase?

**<Replace this line with your answer!>**

## Task 1: Comparison of Effects

Let's compute which of the two variables, Steep Time or Temperature, has a large effect on the overall concentration. To do this, we will use the correlation coefficient $r$, which is a measure of how close the points are to a straight line. So, if $r$ is a larger value, the correlation between the two variables is stronger.

Take a look at how the Black Tea's correlations are found, and then copy that pattern for the Green Tea.

In [ ]:
BT_steep_corr = correlation(black_tea['Steep Time'], black_tea['Concentration (M)'])

BT_temp_corr = correlation(black_tea['Temperature'], black_tea['Concentration (M)'])

BT_test_stat = BT_steep_corr - BT_temp_corr

print(f"The Correlation Coefficient between the Steep Time and Concentration of Black Tea is {BT_steep_corr}")

print(f"The Correlation Coefficient between the Temperature and Concentration of Black Tea is {BT_temp_corr}")

print(f"The difference between these two correlations is {BT_test_stat}")

In [ ]:
# your turn!

GT_steep_corr = correlation(..., ...)

GT_temp_corr = correlation(..., ...)

GT_test_stat = ...

print(f"The Correlation Coefficient between the Steep Time and Concentration of Green Tea is {GT_steep_corr}")

print(f"The Correlation Coefficient between the Temperature and Concentration of Green Tea is {GT_temp_corr}")

print(f"The difference between these two correlations is {GT_test_stat}")

Evaluate the Correlations. For each type of tea, what variable seems to have the greater effect on the Concentration? If you had to guess, why would this be the case?

**<Replace this with your answer!>**

Now, let's see whether the difference in correlation is actually significant. Since we're studying two types of teas, it's best we made a custom function.

There are two lines in that code you'll need to fill out.

In [ ]:
def correlation_sim(table, header_x1, header_x2, header_y):

    """Does simulations of the correlations between header_x1 & header_y, and header_x2 & header_y."""

    test_stats = []

    test_data = table.select(header_x1, header_x2, header_y)

    for i in np.arange(1000):

        random.shuffle(test_data[header_y])
    
        varx1 = test_data[header_x1]
        varx2 = test_data[header_x2]
        vary = test_data[header_y]

        # get the correlation between varx1 and vary here
    
        corr1 = ...

        # get the correlation between varx2 and vary here
    
        corr2 = ...

        test_stats.append(corr1-corr2)

    return np.array(test_stats)

Now, use your function to simulate for green tea. The simulations of black tea have been done for you.

In [ ]:
black_tea_sims = correlation_sim(black_tea, 'Steep Time', 'Temperature', 'Concentration (M)')

green_tea_sims = correlation_sim(..., ..., ..., ...)

Now, let's look at our simulation results.

In [ ]:
plt.hist(black_tea_sims, bins = 30, color = 'blue', label = 'randomized differences in correlation')

plt.scatter(BT_test_stat, 0, color = 'red', label = 'actual difference in correlation')

plt.legend()

plt.show()

In [ ]:
BT_p_value = np.count_nonzero(black_tea_sims >= BT_test_stat)/len(black_tea_sims)
BT_p_value

In [ ]:
plt.hist(green_tea_sims, bins = 30, color = 'blue', label = 'randomized differences in correlation')

plt.scatter(GT_test_stat, 0, color = 'red', label = 'actual difference in correlation')

In [ ]:
GT_p_value = np.count_nonzero(green_tea_sims >= GT_test_stat)/len(green_tea_sims)
GT_p_value

Evaluate the correlations based on the p-values we have obtained. What can we say about the effects of Temperature and Steep Time on Concentration?

**<Replace this with your answer!>**

## Task 2: Predicting the Concentration for any Steep Time and Temperature
Take a look at this gradient mapping below. The y-axis is the steep time, and x-axis is the temperature. Python uses the measurements we have made to color the graph according to the scale to the right. 

The maps are populated with small black dots, which represent our measurements. You'll see that we have only carried out an extremely small analysis of the possible steep times and temperatures. However, it is still wide enough that we can predict the concentration for any steep time/temperature.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

bt = black_tea

x = bt["Temperature"]
y = bt["Steep Time"]
z = bt["Concentration (M)"]

# Fine grid
xi = np.linspace(min(x), max(x), 200)
yi = np.linspace(min(y), max(y), 200)
XI, YI = np.meshgrid(xi, yi)

# Interpolate
ZI = griddata((x, y), z, (XI, YI), method='cubic')

plt.figure(figsize=(7,6))
plt.imshow(
    ZI,
    extent=[x.min()-10, x.max()+10, y.min()-1, y.max()+1],
    origin='lower',
    aspect='auto',
    cmap='viridis'
)

plt.scatter(x, y, c='k', s=20)  # show actual measurements
plt.xlabel("Temperature (°C)")
plt.ylabel("Steep Time (min)")
plt.colorbar(label="Concentration (M)")
plt.show()

In [ ]:
gt = green_tea

x = gt["Temperature"]
y = gt["Steep Time"]
z = gt["Concentration (M)"]

# Fine grid
xi = np.linspace(min(x), max(x), 200)
yi = np.linspace(min(y), max(y), 200)
XI, YI = np.meshgrid(xi, yi)

# Interpolate
ZI = griddata((x, y), z, (XI, YI), method='cubic')

plt.figure(figsize=(7,6))
plt.imshow(
    ZI,
    extent=[x.min()-10, x.max()+10, y.min()-1, y.max()+1],
    origin='lower',
    aspect='auto',
    cmap='viridis'
)

plt.scatter(x, y, c='k', s=20)  # show actual measurements
plt.xlabel("Temperature (°C)")
plt.ylabel("Steep Time (min)")
plt.colorbar(label="Concentration (M)")
plt.show()

The method we will use to predict the concentration for any tea is called **k Nearest Neighbors**. 

First, however, we need to separate the green tea data into two portions: training data and test data. The former will be used to do predict the concentration, and the latter will be used to test how accurate our method is.

In [ ]:
# 80% will be designated as training, 20% as test

training, test = green_tea.split(round(green_tea.num_rows*0.8)) 
training

Let's say we want to find the concentration when Steep Time is 5 minutes and Temperature is 83°C. We want to get the rows that are nearest to that point, and we will do so via the distance formula.

$$dist = \sqrt{(temp - 83)^2+(steeptime - 5)^2}$$

In [ ]:
temp = training[2]

steeptime = training[1]

dist = ...

prediction_table = training.with_column('Distance from target', dist)
prediction_table

Sort the table by our newly-added column. Which row is the shortest distance away from the target?

**<Replace this with your answer!>**

In k Nearest Neighbors, we won't use just the closest row, but the average concentrations of some number of closest rows. This cell below gets the average of the three closest rows.

In [ ]:
np.mean(prediction_table.sort('Distance from target').take(np.arange(3)).column('Concentration (M)'))

Now, let's automate it for any time and any temperature.

In [ ]:
def predict_conc(training, targetTime, targetTemp, k):

    """Finds the k nearest neighbors closest to the targetTime and targetTemp,
    and then gets their average concentration."""

    temp = training[2]

    steeptime = training[1]

    # copy and paste your distance code from above (but replace 5 and 83 with targetTime and targetTemp)

    dist = ...

    prediction_table = training.with_column('Distance from target', dist)

    return np.mean(prediction_table.sort('Distance from target').take(np.arange(k)).column('Concentration (M)'))

Wonderful! Now we can test our methods with the `test` data. Namely, we will predict the concentration using those test points' time and temperature, and then compare it with the actual concentration values. This comparison (predicted - actual) is stored in `errors`.

How we compare is through the RMSE, or Root Mean Squared Error. We find it by squaring all the `errors`, getting the mean of the squared errors, and then taking the square root.

In [ ]:
RMSE = []
for k in np.arange(1, 10):

    errors = []
    
    for i in np.arange(test.num_rows):
        time = test[1][i]
        temp = test[2][i]
    
        actual_conc = test[4][i]
    
        predicted_conc = predict_conc(training, time, temp, k)
    
        errors.append(predicted_conc - actual_conc)

    RMSE.append(np.sqrt(np.mean([x**2 for x in errors])))

In [ ]:
plt.scatter(np.arange(1, 10), RMSE)

When we add more and more neighbors into the final prediction, what happens to our error?

**<Replace this with your answer!>**